# Carbon Majors — Ingest & Exploration

Load, validate, and explore the Carbon Majors high-granularity dataset (InfluenceMap, 2026 release, covering 1854–2024).

**Outputs**
- `data/processed/cm_entity_year.parquet` — emissions aggregated by entity × year (summed across commodities)
- `data/processed/cm_cumulative_summary.parquet` — cumulative totals and global share per entity
- `data/processed/cm_global_annual.parquet` — global annual totals by parent type

**Attribution chain position**: this notebook covers the first step — *Named Emitter → Cumulative Emissions*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

RAW  = Path("../../data/raw/carbon_majors")
PROC = Path("../../data/processed")
PROC.mkdir(exist_ok=True)
FIGS = Path("../../outputs/figures")

## 1. Load & validate

In [ ]:
df = pd.read_csv(RAW / "emissions_high_granularity.csv")

print(f"Shape: {df.shape}")
print(f"Years: {df.year.min()}–{df.year.max()}")
print(f"Entities (parent): {df.parent_entity.nunique()}")
print(f"Reporting entities: {df.reporting_entity.nunique()}")
df.dtypes

In [ ]:
# Null audit
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts[null_counts > 0].to_string())
print()

# Sanity check: total_emissions should equal product + operational
computed = df["product_emissions_MtCO2"] + df["total_operational_emissions_MtCO2e"]
discrepancy = (df["total_emissions_MtCO2e"] - computed).abs()
print(f"Max discrepancy (product + operational vs total): {discrepancy.max():.6f} MtCO2e")
print(f"Rows with discrepancy > 0.001: {(discrepancy > 0.001).sum()}")

In [ ]:
# Cross-check against published total: ~1,421 GtCO2e cumulative through 2022
total_cumulative = df[df.year <= 2022]["total_emissions_MtCO2e"].sum() / 1000  # convert to GtCO2e
print(f"Cumulative total 1854–2022: {total_cumulative:.1f} GtCO2e")
print(f"Expected (launch report):   1421.0 GtCO2e")
print(f"Difference: {abs(total_cumulative - 1421):.1f} Gt ({abs(total_cumulative - 1421)/1421*100:.2f}%)")

## 2. Global annual emissions

How do total emissions break down by entity type (state-owned, investor-owned, nation-state) across time?

In [ ]:
annual_by_type = (
    df.groupby(["year", "parent_type"])["total_emissions_MtCO2e"]
    .sum()
    .unstack("parent_type")
    .fillna(0)
)

# Reorder columns for logical stacking
col_order = [c for c in ["Investor-owned Company", "State-owned Entity", "Nation State"] if c in annual_by_type.columns]
annual_by_type = annual_by_type[col_order]

fig, ax = plt.subplots(figsize=(12, 5))
annual_by_type.plot.area(ax=ax, alpha=0.85)
ax.set_title("Annual emissions from Carbon Majors by entity type, 1854–2024", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("MtCO₂e / year")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.axvline(1988, color="red", linestyle="--", alpha=0.6, label="1988 (Hansen testimony)")
ax.axvline(2015, color="orange", linestyle="--", alpha=0.6, label="2015 (Paris Agreement)")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "cm_annual_by_type.png", bbox_inches="tight")
plt.show()
print(f"\n2024 total: {annual_by_type.loc[2024].sum():,.0f} MtCO2e")
print(annual_by_type.tail(3))

## 3. Top emitters — cumulative totals

Who are the largest contributors across the full historical record? We compute cumulative totals for three windows:
- **All time** (1854–2024) — relevant for total atmospheric forcing
- **Post-1988** — relevant for "knew or should have known" legal framing
- **Post-Paris** (2016–2024) — relevant for Paris-era commitments

In [ ]:
def cumulative_by_entity(df, year_min=None, year_max=None, col="total_emissions_MtCO2e"):
    mask = pd.Series(True, index=df.index)
    if year_min is not None:
        mask &= df["year"] >= year_min
    if year_max is not None:
        mask &= df["year"] <= year_max
    return (
        df[mask]
        .groupby(["parent_entity", "parent_type"])[col]
        .sum()
        .reset_index()
        .rename(columns={col: "cumulative_MtCO2e"})
        .sort_values("cumulative_MtCO2e", ascending=False)
        .reset_index(drop=True)
    )

cumul_all    = cumulative_by_entity(df)
cumul_post88 = cumulative_by_entity(df, year_min=1988)
cumul_paris  = cumulative_by_entity(df, year_min=2016)

global_total = cumul_all["cumulative_MtCO2e"].sum()
cumul_all["share_pct"] = cumul_all["cumulative_MtCO2e"] / global_total * 100
cumul_all["cumulative_GtCO2e"] = cumul_all["cumulative_MtCO2e"] / 1000

print(f"Global cumulative total: {global_total/1000:.1f} GtCO2e")
print(f"\nTop 10 emitters (all time):")
print(cumul_all[["parent_entity", "parent_type", "cumulative_GtCO2e", "share_pct"]].head(10).to_string(index=False))

In [ ]:
top20 = cumul_all.head(20).copy()

# Color by entity type
type_colors = {
    "Investor-owned Company": "#2196F3",
    "State-owned Entity":     "#FF5722",
    "Nation State":           "#4CAF50",
}
colors = top20["parent_type"].map(type_colors)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top20["parent_entity"][::-1], top20["cumulative_GtCO2e"][::-1], color=colors[::-1])
ax.set_xlabel("Cumulative emissions (GtCO₂e, 1854–2024)")
ax.set_title("Top 20 Carbon Majors — cumulative historical emissions", fontsize=13)

# Add share labels
for bar, (_, row) in zip(bars[::-1], top20.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{row.share_pct:.1f}%", va="center", fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "cm_top20_cumulative.png", bbox_inches="tight")
plt.show()

In [ ]:
# Cumulative share: how many entities account for 50%, 75%, 90%?
cumul_all["cumulative_share_pct"] = cumul_all["share_pct"].cumsum()
for threshold in [50, 75, 90]:
    n = (cumul_all["cumulative_share_pct"] <= threshold).sum() + 1
    print(f"Top {n:3d} entities account for {threshold}% of cumulative emissions")

## 4. Cumulative emissions over time — top entities

When did each major emitter's contribution accumulate? This matters for the attribution chain: earlier emissions have had longer to force warming.

In [ ]:
top10_names = cumul_all.head(10)["parent_entity"].tolist()

# Annual emissions for top 10, then cumsum per entity
top10_annual = (
    df[df["parent_entity"].isin(top10_names)]
    .groupby(["year", "parent_entity"])["total_emissions_MtCO2e"]
    .sum()
    .unstack("parent_entity")
    .fillna(0)
    .sort_index()
)
top10_cumul = top10_annual.cumsum() / 1000  # GtCO2e

fig, ax = plt.subplots(figsize=(12, 6))
top10_cumul.plot(ax=ax, linewidth=1.5)
ax.set_title("Cumulative emissions — top 10 Carbon Majors, 1854–2024", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Cumulative GtCO₂e")
ax.axvline(1988, color="red", linestyle="--", alpha=0.5, label="1988")
ax.axvline(2015, color="orange", linestyle="--", alpha=0.5, label="Paris 2015")
ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGS / "cm_top10_cumulative_timeseries.png", bbox_inches="tight")
plt.show()

## 5. Commodity breakdown

What fraction of total emissions comes from each commodity? Important for understanding which emission types dominate and how to weight CH4 vs CO2.

In [ ]:
# Aggregate coal types for readability
coal_types = [c for c in df["commodity"].unique() if "Coal" in c]
df["commodity_group"] = df["commodity"].replace({c: "Coal" for c in coal_types})

commodity_totals = (
    df.groupby("commodity_group")["total_emissions_MtCO2e"]
    .sum()
    .sort_values(ascending=False)
)
commodity_pct = (commodity_totals / commodity_totals.sum() * 100).round(1)

print("Cumulative emissions by commodity:")
for name, pct in commodity_pct.items():
    print(f"  {name:<20} {pct:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie
commodity_totals.plot.pie(ax=axes[0], autopct="%1.1f%%", startangle=90, fontsize=9)
axes[0].set_title("Share by commodity (cumulative)")
axes[0].set_ylabel("")

# Annual by commodity group
annual_commodity = (
    df.groupby(["year", "commodity_group"])["total_emissions_MtCO2e"]
    .sum()
    .unstack("commodity_group")
    .fillna(0)
)
annual_commodity.plot.area(ax=axes[1], alpha=0.8)
axes[1].set_title("Annual emissions by commodity")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("MtCO₂e / year")
axes[1].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "cm_commodity_breakdown.png", bbox_inches="tight")
plt.show()

## 6. Scope 1 vs Scope 3 split

Product emissions (scope 3 — combustion by end users) vs operational emissions (scope 1 — flaring, venting, fugitive methane). The scope split matters for legal liability: scope 3 attribution is more contested.

In [ ]:
scope_annual = df.groupby("year").agg(
    scope3_product  = ("product_emissions_MtCO2", "sum"),
    scope1_operational = ("total_operational_emissions_MtCO2e", "sum"),
)

scope_totals = scope_annual.sum() / 1000
print("Cumulative totals (GtCO2e):")
print(f"  Scope 3 (product/combustion):  {scope_totals.scope3_product:.1f} Gt  ({scope_totals.scope3_product/scope_totals.sum()*100:.1f}%)")
print(f"  Scope 1 (operational):         {scope_totals.scope1_operational:.1f} Gt  ({scope_totals.scope1_operational/scope_totals.sum()*100:.1f}%)")

fig, ax = plt.subplots(figsize=(12, 4))
scope_annual.plot.area(ax=ax, alpha=0.8, color=["#1976D2", "#F57C00"])
ax.set_title("Annual emissions: scope 3 (product combustion) vs scope 1 (operational)")
ax.set_xlabel("Year")
ax.set_ylabel("MtCO₂e / year")
ax.legend(["Scope 3 — product combustion", "Scope 1 — operational"], fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "cm_scope_split.png", bbox_inches="tight")
plt.show()

## 7. Three-window comparison: all-time, post-1988, post-Paris

The legal framing often changes which time window is used. Here we compare the top 15 emitters across all three windows to see how rankings shift.

In [ ]:
top15_names = cumul_all.head(15)["parent_entity"].tolist()

windows = {
    "All time\n(1854–2024)": cumulative_by_entity(df),
    "Post-1988": cumulative_by_entity(df, year_min=1988),
    "Post-Paris\n(2016–2024)": cumulative_by_entity(df, year_min=2016),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
for ax, (label, wdf) in zip(axes, windows.items()):
    top = wdf[wdf["parent_entity"].isin(top15_names)].set_index("parent_entity")
    # Reorder by all-time rank
    top = top.loc[[e for e in top15_names if e in top.index]]
    colors = top["parent_type"].map(type_colors)
    ax.barh(range(len(top)), top["cumulative_MtCO2e"] / 1000, color=colors)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top.index, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("GtCO₂e")

legend_elements = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Top 15 emitters across different attribution windows", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGS / "cm_three_windows.png", bbox_inches="tight")
plt.show()

## 8. Save processed outputs

In [ ]:
# Entity × year (summed across commodities) — primary input for attribution notebooks
entity_year = (
    df.groupby(["year", "parent_entity", "parent_type", "lei"])
    .agg(
        product_emissions_MtCO2           = ("product_emissions_MtCO2", "sum"),
        flaring_emissions_MtCO2           = ("flaring_emissions_MtCO2", "sum"),
        venting_emissions_MtCO2           = ("venting_emissions_MtCO2", "sum"),
        own_fuel_use_emissions_MtCO2      = ("own_fuel_use_emissions_MtCO2", "sum"),
        fugitive_methane_emissions_MtCO2e = ("fugitive_methane_emissions_MtCO2e", "sum"),
        total_operational_emissions_MtCO2e= ("total_operational_emissions_MtCO2e", "sum"),
        total_emissions_MtCO2e            = ("total_emissions_MtCO2e", "sum"),
    )
    .reset_index()
)

# Cumulative summary with global share (three windows)
cumul_summary = cumul_all.copy()
cumul_summary = cumul_summary.merge(
    cumul_post88[["parent_entity", "cumulative_MtCO2e"]].rename(columns={"cumulative_MtCO2e": "cumul_post1988_MtCO2e"}),
    on="parent_entity", how="left"
).merge(
    cumul_paris[["parent_entity", "cumulative_MtCO2e"]].rename(columns={"cumulative_MtCO2e": "cumul_post_paris_MtCO2e"}),
    on="parent_entity", how="left"
)

# Global annual
global_annual = df.groupby(["year", "parent_type"])["total_emissions_MtCO2e"].sum().reset_index()

entity_year.to_parquet(PROC / "cm_entity_year.parquet", index=False)
cumul_summary.to_parquet(PROC / "cm_cumulative_summary.parquet", index=False)
global_annual.to_parquet(PROC / "cm_global_annual.parquet", index=False)

print("Saved:")
print(f"  cm_entity_year.parquet       {len(entity_year):>6,} rows")
print(f"  cm_cumulative_summary.parquet {len(cumul_summary):>5,} rows")
print(f"  cm_global_annual.parquet      {len(global_annual):>5,} rows")

## Key findings

Update this cell after running the notebook.

- **Total covered**: ___ GtCO₂e cumulative 1854–2024
- **Top entity**: ___ with ___% of global total
- **Concentration**: top ___ entities = 50% of all emissions
- **Scope split**: ~___% scope 3 (product combustion), ~___% scope 1 (operational)
- **Post-1988 share**: ___% of all-time emissions occurred after Hansen's 1988 testimony

→ See `wiki/findings/2026-05-15-carbon-majors-ingest.md` for the full write-up.